# ARC3 CPU tokenizer shape smoke

Uses the actual attached AutoTokenizer and pinned Pennyroyal tokenizer wheels to
check the transport probe's 2K, 8K and 16K payload counts. It records default
chat-template return types, mapping keys, input-ID shape, actual counts, package
versions and helper SHA256. Explicit flat-token rendering independently checks
the helper's count against the real tokenizer.

CPU only; no inference server, HTTP call, game execution, model weights or
submission artifact. This validates tokenizer compatibility, not gameplay score
or GPU transport performance. Required light dependencies reuse the CPU image
or compatible attached offline wheels. Heavy inference packages are refused.


In [ ]:
import hashlib
import json
import os
from pathlib import Path
import subprocess
import sys

if os.environ.get('KAGGLE_IS_COMPETITION_RERUN', '').strip().lower() in {'1', 'true'}:
    raise RuntimeError('CPU tokenizer diagnostic must not run as a competition submission')
work = Path('/kaggle/working/cpu_tokenizer_smoke')
work.mkdir(parents=True, exist_ok=True)
model_candidates = [
    Path('/kaggle/input/models/dfranzen/intel-qwen3.8-flash-next-w4a16-autoround/transformers/default/1'),
    Path('/kaggle/input/intel-qwen3.8-flash-next-w4a16-autoround/transformers/default/1'),
]
models = list(dict.fromkeys(path.resolve() for path in model_candidates if (path/'tokenizer.json').is_file()))
if len(models) != 1:
    raise RuntimeError(f'Expected exactly one attached model tokenizer, found {models}')
wheel_roots = []
for root in [Path('/kaggle/input/datasets/dfranzen/pennyroyal-v253'), Path('/kaggle/input/pennyroyal-v253')]:
    if root.is_dir():
        for parent in [root, *root.rglob('wheels')]:
            if parent.is_dir() and list(parent.glob('transformers-*.whl')):
                wheel_roots.append(parent.resolve())
wheel_roots = list(dict.fromkeys(wheel_roots))
if len(wheel_roots) != 1:
    raise RuntimeError(f'Expected exactly one pinned Pennyroyal wheel directory, found {wheel_roots}')
installer = work/'install_cpu_tokenizer.py'
runner = work/'run_cpu_tokenizer_smoke.py'
helper_path = work/'load_parity_probe.py'
installer.write_text('from __future__ import annotations\nimport email.parser\nimport importlib.metadata as metadata\nimport json\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport zipfile\n\ntry:\n    from packaging.requirements import Requirement\n    from packaging.tags import sys_tags\n    from packaging.utils import canonicalize_name, parse_wheel_filename\n    from packaging.version import Version\nexcept ImportError:\n    from pip._vendor.packaging.requirements import Requirement\n    from pip._vendor.packaging.tags import sys_tags\n    from pip._vendor.packaging.utils import canonicalize_name, parse_wheel_filename\n    from pip._vendor.packaging.version import Version\n\nroot = Path(sys.argv[1])\ntarget = Path(sys.argv[2])\nreceipt = Path(sys.argv[3])\ncompatible_tags = set(sys_tags())\nforbidden = {\'torch\', \'torchvision\', \'torchaudio\', \'tensorflow\', \'flax\', \'jax\', \'jaxlib\',\n             \'sglang\', \'triton\', \'flashinfer\', \'flashinfer-python\', \'vllm\', \'cuda-python\'}\n\ndef allowed(name):\n    name = canonicalize_name(name)\n    return name not in forbidden and not name.startswith((\'nvidia-\', \'cuda-\', \'flashinfer-\', \'torch-\'))\n\nall_wheels = []\nfor path in sorted(root.glob(\'*.whl\')):\n    try:\n        name, version, build, tags = parse_wheel_filename(path.name)\n    except Exception:\n        continue\n    if compatible_tags.intersection(tags):\n        all_wheels.append((canonicalize_name(name), version, path))\n\nplan = {}\nrequirements = []\nprocessed = set()\n\ndef choose(name, specifier=None, *, pinned=False):\n    name = canonicalize_name(name)\n    if not allowed(name):\n        raise RuntimeError(\'Refusing heavyweight/GPU dependency: \' + name)\n    candidates = [(version, path) for n, version, path in all_wheels\n                  if n == name and (specifier is None or specifier.contains(version, prereleases=True))]\n    if not candidates:\n        raise RuntimeError(f\'No compatible offline wheel for {name}{specifier or ""}\')\n    if pinned and len(candidates) != 1:\n        raise RuntimeError(f\'Expected one pinned offline {name} wheel, found {len(candidates)}\')\n    version, path = max(candidates, key=lambda item: (item[0], item[1].name))\n    current = plan.get(name)\n    if current is not None and current[0] != version:\n        raise RuntimeError(f\'Offline requirement conflict for {name}: {current[0]} versus {version}\')\n    plan[name] = (version, path)\n    return path\n\nfor package in (\'transformers\', \'tokenizers\', \'huggingface-hub\'):\n    choose(package, pinned=True)\n\nimage_dependencies = {}\nwhile True:\n    pending = [(name, value) for name, value in plan.items() if name not in processed]\n    if not pending:\n        break\n    for name, (version, path) in pending:\n        processed.add(name)\n        with zipfile.ZipFile(path) as wheel:\n            names = [member for member in wheel.namelist() if member.endswith(\'.dist-info/METADATA\')]\n            if len(names) != 1:\n                raise RuntimeError(\'Expected one wheel metadata file: \' + path.name)\n            message = email.parser.Parser().parsestr(wheel.read(names[0]).decode())\n        for raw in message.get_all(\'Requires-Dist\', []):\n            req = Requirement(raw)\n            if req.marker is not None and not req.marker.evaluate({\'extra\': \'\'}):\n                continue\n            dep = canonicalize_name(req.name)\n            if req.url:\n                raise RuntimeError(\'Direct URL dependency is not permitted: \' + raw)\n            requirements.append((name, raw))\n            if dep in plan:\n                if not req.specifier.contains(plan[dep][0], prereleases=True):\n                    raise RuntimeError(f\'Pinned wheel requirement conflict: {name} requires {raw}\')\n                continue\n            try:\n                installed = metadata.version(dep)\n            except metadata.PackageNotFoundError:\n                installed = None\n            if installed is not None and req.specifier.contains(installed, prereleases=True):\n                image_dependencies[dep] = installed\n                continue\n            choose(dep, req.specifier)\n\n# Verify all collected constraints after the closure is complete.\nfor owner, raw in requirements:\n    req = Requirement(raw)\n    dep = canonicalize_name(req.name)\n    version = plan[dep][0] if dep in plan else image_dependencies[dep]\n    if not req.specifier.contains(version, prereleases=True):\n        raise RuntimeError(f\'Offline dependency conflict: {owner} requires {raw}; selected {version}\')\n\ntarget.mkdir(parents=True, exist_ok=True)\ninstall = [sys.executable, \'-m\', \'pip\', \'install\', \'--no-index\', \'--no-deps\', \'--no-compile\',\n           \'--disable-pip-version-check\', \'--no-warn-script-location\', \'--target\', str(target)]\ninstall += [str(value[1]) for name, value in sorted(plan.items())]\nsubprocess.run(install, check=True, timeout=180)\nreport = {\'purpose\': \'CPU tokenizer-only offline dependency preparation\',\n          \'python\': sys.version, \'target_directory\': str(target),\n          \'pinned_tokenizer_packages\': [\'transformers\', \'tokenizers\', \'huggingface-hub\'],\n          \'installed_wheels\': {name: {\'version\': str(version), \'wheel\': path.name}\n                               for name, (version, path) in sorted(plan.items())},\n          \'reused_image_dependencies\': dict(sorted(image_dependencies.items())),\n          \'gpu_packages_installed\': False, \'model_weights_loaded\': False,\n          \'server_started\': False, \'http_requests_sent\': False}\nreceipt.write_text(json.dumps(report, indent=2) + \'\\n\')\nprint(json.dumps(report, indent=2), flush=True)\n')
runner.write_text("from __future__ import annotations\nimport collections.abc\nimport hashlib\nimport importlib.metadata as metadata\nimport importlib.util\nimport json\nimport os\nfrom pathlib import Path\nimport sys\nimport time\n\nstarted = time.monotonic()\nsys.path.insert(0, sys.argv[1])\nmodel_path, helper_path, output_path = map(Path, sys.argv[2:5])\nreport = {'scope': 'CPU-only real-tokenizer shape smoke; no model inference or score evidence',\n          'passed': False, 'weights_loaded': False, 'server_started': False,\n          'http_requests_sent': False, 'gpu_enabled': False, 'official_score': None,\n          'authorizes_submission': False,\n          'helper_sha256': hashlib.sha256(helper_path.read_bytes()).hexdigest(),\n          'model_path': str(model_path), 'python': sys.version, 'cases': []}\n\ndef token_shape(value):\n    info = {'type': type(value).__module__ + '.' + type(value).__name__}\n    if isinstance(value, collections.abc.Mapping):\n        info['keys'] = sorted(map(str, value))\n        info['mapping_length'] = len(value)\n        if 'input_ids' not in value:\n            raise RuntimeError('Template mapping has no input_ids')\n        value = value['input_ids']\n        info['input_ids_type'] = type(value).__module__ + '.' + type(value).__name__\n    if hasattr(value, 'tolist'):\n        value = value.tolist()\n    if not isinstance(value, (list, tuple)):\n        raise RuntimeError('Unsupported token sequence: ' + str(type(value)))\n    if len(value) and isinstance(value[0], (list, tuple)):\n        info['shape'] = [len(value), len(value[0])]\n        if len(value) != 1:\n            raise RuntimeError('Expected one rendered sequence')\n        value = value[0]\n    else:\n        info['shape'] = [len(value)]\n    if not all(isinstance(item, int) and not isinstance(item, bool) for item in value):\n        raise RuntimeError('Rendered input_ids contain noninteger IDs')\n    info['token_count'] = len(value)\n    return info\n\ntry:\n    from transformers import AutoTokenizer\n    import transformers\n    import tokenizers\n    import huggingface_hub\n    report['package_versions'] = {name: metadata.version(name) for name in\n        ('transformers', 'tokenizers', 'huggingface-hub', 'jinja2')}\n    report['package_paths'] = {name: module.__file__ for name, module in\n        [('transformers', transformers), ('tokenizers', tokenizers), ('huggingface_hub', huggingface_hub)]}\n    tokenizer = AutoTokenizer.from_pretrained(str(model_path), local_files_only=True, trust_remote_code=True)\n    report['tokenizer_class'] = type(tokenizer).__module__ + '.' + type(tokenizer).__name__\n    report['tokenizer_is_fast'] = bool(getattr(tokenizer, 'is_fast', False))\n    report['chat_template_type'] = type(getattr(tokenizer, 'chat_template', None)).__name__\n    spec = importlib.util.spec_from_file_location('cpu_tokenizer_probe_helper', helper_path)\n    helper = importlib.util.module_from_spec(spec)\n    sys.modules[spec.name] = helper\n    spec.loader.exec_module(helper)\n    for index, target in enumerate((2048, 8192, 16384)):\n        case = {'case': index, 'target_input_tokens': target, 'max_output_tokens': 64, 'seed': 20261017 + index}\n        payload, helper_count = helper.build_payload(tokenizer, case, 0, 'flashnext', 25)\n        kwargs = dict(tokenize=True, add_generation_prompt=True, enable_thinking=True, preserve_thinking=True)\n        default_shape = token_shape(tokenizer.apply_chat_template(payload['messages'], **kwargs))\n        flat_shape = token_shape(tokenizer.apply_chat_template(payload['messages'], return_dict=False, **kwargs))\n        actual = flat_shape['token_count']\n        valid = (helper_count == actual == default_shape['token_count'] and abs(actual-target) <= 8)\n        row = {'target_input_tokens': target, 'helper_token_count': helper_count,\n               'default_template': default_shape, 'explicit_flat_template': flat_shape,\n               'max_output_tokens': payload['max_tokens'], 'passed': valid}\n        report['cases'].append(row)\n        print(json.dumps(row), flush=True)\n        if not valid:\n            raise RuntimeError(f'Real tokenizer input-length mismatch for {target}: helper={helper_count}, actual={actual}')\n    loaded_frameworks = sorted(name for name in ('torch', 'tensorflow', 'jax', 'flax') if name in sys.modules)\n    report['loaded_ml_frameworks'] = loaded_frameworks\n    # A tokenizer dependency may import an existing framework incidentally.\n    # Reading this flag does not initialize CUDA or probe device availability.\n    torch_module = sys.modules.get('torch')\n    torch_cuda = getattr(torch_module, 'cuda', None)\n    report['torch_cuda_initialized'] = bool(torch_cuda.is_initialized()) if torch_cuda is not None else False\n    if report['torch_cuda_initialized']:\n        raise RuntimeError('CPU tokenizer smoke observed an initialized CUDA runtime')\n    report['passed'] = len(report['cases']) == 3 and all(row['passed'] for row in report['cases'])\nexcept BaseException as exc:\n    report['error_type'] = type(exc).__name__\n    report['error'] = str(exc)\n    raise\nfinally:\n    report['elapsed_seconds'] = time.monotonic() - started\n    output_path.write_text(json.dumps(report, indent=2, allow_nan=False) + '\\n')\n    print(json.dumps({key: value for key, value in report.items() if key != 'cases'}, indent=2), flush=True)\n")
helper_path.write_text('"""Bounded transport/cache probe; never starts a server or touches a game gateway.\n\nThis measures 25 versus 110 pending client states behind ten HTTP slots. It does\nnot reproduce D-prime scheduling, gameplay, image preprocessing, or hidden score.\nRun only in an isolated subprocess, against the already loaded local server.\n"""\nfrom __future__ import annotations\n\nimport argparse\nfrom collections.abc import Mapping\nfrom concurrent.futures import ThreadPoolExecutor, wait\nfrom dataclasses import asdict, dataclass\nimport hashlib\nimport json\nfrom numbers import Integral\nfrom pathlib import Path\nimport threading\nimport time\nfrom urllib.parse import urlsplit\n\n\n@dataclass(frozen=True)\nclass ProbeConfig:\n    populations: tuple[int, ...] = (25, 110, 110, 25)\n    active_slots: int = 10\n    requests_per_phase: int = 64\n    input_tokens: tuple[int, ...] = (2048, 8192, 16384)\n    max_output_tokens: int = 64\n    phase_seconds: float = 60.0\n    request_timeout_seconds: float = 20.0\n    seed: int = 20261017\n\n    def validate(self):\n        if self.populations != (25, 110, 110, 25):\n            raise ValueError(\'Use the fixed ABBA population order for this diagnostic\')\n        if self.active_slots != 10:\n            raise ValueError(\'Preserve the existing ten running requests\')\n        if not 61 <= self.requests_per_phase <= 128:\n            raise ValueError(\'Use61..128 requests to cross the pinned60-state Mamba cache\')\n        if max(self.input_tokens) > 114688 or min(self.input_tokens) < 1024:\n            raise ValueError(\'Probe prompts must stay within the pinned context\')\n        if not 1 <= self.max_output_tokens <= 1024:\n            raise ValueError(\'Bound output tokens to1..1024\')\n        if not 0 < self.request_timeout_seconds <= self.phase_seconds <= 90:\n            raise ValueError(\'Request/phase deadlines must be positive and bounded\')\n        return self\n\n\ndef local_url(base_url):\n    parsed = urlsplit(base_url)\n    if parsed.scheme != \'http\' or parsed.hostname not in {\'127.0.0.1\', \'localhost\', \'::1\'}:\n        raise ValueError(\'The probe may call only the already running loopback HTTP server\')\n    if parsed.username or parsed.password or parsed.query or parsed.fragment:\n        raise ValueError(\'Credentials, queries and fragments are not permitted\')\n    return base_url.rstrip(\'/\')\n\n\ndef quantile(values, fraction):\n    values = sorted(values)\n    if not values:\n        return None\n    index = (len(values) - 1) * fraction\n    lo, hi = int(index), min(len(values) - 1, int(index) + 1)\n    return values[lo] * (hi - index) + values[hi] * (index - lo) if lo != hi else values[lo]\n\n\ndef shape_plan(config):\n    """The same admitted request lengths/count/output cap in every phase."""\n    config.validate()\n    return [dict(case=i, target_input_tokens=config.input_tokens[i % len(config.input_tokens)],\n                 max_output_tokens=config.max_output_tokens,\n                 seed=config.seed + i) for i in range(config.requests_per_phase)]\n\n\ndef chat_template_token_count(rendered):\n    """Count one tokenized conversation across Transformers return shapes.\n\n    Recent tokenizers can return a BatchEncoding mapping with ``input_ids``\n    and ``attention_mask`` even without requesting tensors. Counting mapping\n    fields (or a batch dimension) silently reports two (or one) tokens. Accept\n    flat IDs and a single batch, including tensor/array values via ``tolist``;\n    reject multiple conversations rather than treating them as token IDs.\n    """\n    if isinstance(rendered, Mapping):\n        if \'input_ids\' not in rendered:\n            raise ValueError(\'Tokenized chat-template mapping is missing input_ids\')\n        rendered = rendered[\'input_ids\']\n    if callable(getattr(rendered, \'tolist\', None)):\n        rendered = rendered.tolist()\n    if not isinstance(rendered, (list, tuple)):\n        raise TypeError(\'Tokenized chat template must contain an input_ids sequence\')\n    if rendered and isinstance(rendered[0], (list, tuple)):\n        if len(rendered) != 1:\n            raise ValueError(\'Expected one tokenized conversation, received multiple batches\')\n        rendered = rendered[0]\n    if any(isinstance(token, bool) or not isinstance(token, Integral) for token in rendered):\n        raise TypeError(\'Tokenized chat-template input_ids must be a flat integer sequence\')\n    return len(rendered)\n\n\ndef build_payload(tokenizer, case, phase, model, population):\n    """Unique prefix per phase/case avoids second-phase exact-prefix advantage.\n\n    Match the template-rendered token length, not repeated character length.\n    Contents are synthetic and cannot execute anything outside this probe.\n    """\n    target = case[\'target_input_tokens\']\n    nonce = hashlib.sha256(f"arc-load/{phase}/{case[\'case\'] % population}".encode()).hexdigest()\n    system = (\'Synthetic transport measurement. Read the following numbered board notes, \'\n              \'reason briefly about them, and emit a short summary. Do not call tools. \'\n              f\'Independent case {nonce}.\')\n    filler = tokenizer.encode(\' board row column zero one two three four five six seven eight nine\',\n                              add_special_tokens=False)\n    prefix = tokenizer.encode(f\'Independent synthetic state {nonce}.\\n\', add_special_tokens=False)\n    pad = (filler * ((target // len(filler)) + 2))[:target]\n    tokens = prefix + pad\n\n    def messages(n):\n        return [{\'role\': \'system\', \'content\': system},\n                {\'role\': \'user\', \'content\': tokenizer.decode(tokens[:n])}]\n\n    # Template overhead varies slightly with tokenizer. Remove it once and\n    # correct any tokenization boundary changes with a tiny bounded search.\n    n = max(1, target - len(prefix) - 100)\n    msgs = messages(n)\n    rendered = tokenizer.apply_chat_template(msgs, tokenize=True,\n        add_generation_prompt=True, enable_thinking=True, preserve_thinking=True)\n    n = min(len(tokens), max(1, n + target - chat_template_token_count(rendered)))\n    for _ in range(8):\n        msgs = messages(n)\n        actual = chat_template_token_count(tokenizer.apply_chat_template(msgs, tokenize=True,\n            add_generation_prompt=True, enable_thinking=True, preserve_thinking=True))\n        if actual == target:\n            break\n        n = min(len(tokens), max(1, n + target - actual))\n    if abs(actual - target) > 8:\n        raise RuntimeError(f\'Tokenizer could not match prompt length: wanted{target}, got{actual}\')\n    payload = {\'model\': model, \'messages\': msgs, \'max_tokens\': case[\'max_output_tokens\'],\n               \'temperature\': 0.7, \'top_p\': 0.95, \'top_k\': 20, \'seed\': case[\'seed\'],\n               \'stream\': False,\n               \'chat_template_kwargs\': {\'enable_thinking\': True, \'preserve_thinking\': True}}\n    return payload, actual\n\n\ndef run_phase(population, payloads, config, post, *, clock=time.monotonic):\n    """No inference starts after the deadline; all waiting states are drained.\n\n    A socket read timeout is not a whole-request wall timeout, so the notebook\n    must enforce the separate hard subprocess deadline as well.\n    """\n    config.validate()\n    if len(payloads) != config.requests_per_phase:\n        raise ValueError(\'Incorrect fixed request manifest\')\n    start_gate = threading.Barrier(population + 1)\n    slots = threading.BoundedSemaphore(config.active_slots)\n    lock = threading.Lock()\n    start_at = None\n    deadline = None\n    active = 0\n    peak_active = 0\n    rows = []\n\n    def worker(state):\n        nonlocal active, peak_active\n        start_gate.wait(timeout=10)\n        indices = list(range(state, len(payloads), population))\n        # Empty clients still enter the ten-slot waiting population once.\n        for index in indices or [None]:\n            queued_at = clock()\n            acquired = False\n            while clock() < deadline:\n                if slots.acquire(timeout=min(0.1, max(0.001, deadline-clock()))):\n                    acquired = True\n                    break\n            if not acquired:\n                return\n            try:\n                admitted = clock()\n                if admitted >= deadline or index is None:\n                    return\n                with lock:\n                    active += 1\n                    peak_active = max(peak_active, active)\n                payload, template_tokens = payloads[index]\n                row = {\'state\': state, \'case\': index, \'template_tokens\': template_tokens,\n                       \'queue_seconds\': admitted-queued_at, \'http_seconds\': None,\n                       \'completion_tokens\': 0, \'prompt_tokens\': None, \'ok\': False}\n                try:\n                    timeout = max(0.05, min(config.request_timeout_seconds, deadline-clock()))\n                    response = post(payload, timeout)\n                    row.update(response)\n                    prompt = response.get(\'prompt_tokens\')\n                    completion = response.get(\'completion_tokens\')\n                    row[\'usage_valid\'] = (isinstance(prompt, int) and prompt > 0\n                        and isinstance(completion, int) and 0 < completion <= config.max_output_tokens)\n                    row[\'input_length_matched\'] = (isinstance(prompt, int)\n                        and abs(prompt-template_tokens) <= 16)\n                    row[\'ok\'] = bool(response.get(\'ok\')) and row[\'usage_valid\'] and row[\'input_length_matched\']\n                except Exception as exc:\n                    row[\'error_type\'] = type(exc).__name__\n                    row[\'error\'] = str(exc)[:500]\n                finally:\n                    row[\'http_seconds\'] = clock()-admitted\n                    with lock:\n                        active -= 1\n                        rows.append(row)\n            finally:\n                slots.release()\n\n    with ThreadPoolExecutor(max_workers=population, thread_name_prefix=\'load-state\') as executor:\n        futures = [executor.submit(worker, i) for i in range(population)]\n        start_at = clock()\n        deadline = start_at + config.phase_seconds\n        start_gate.wait(timeout=10)\n        wait(futures)\n        for future in futures:\n            future.result()\n    elapsed = clock()-start_at\n    successes = [row for row in rows if row[\'ok\']]\n    return {\'population\': population, \'active_slots\': config.active_slots,\n            \'peak_active_http\': peak_active, \'requested\': len(payloads),\n            \'attempted\': len(rows), \'completed\': len(successes),\n            \'valid_transport_comparison\': len(successes) == len(payloads),\n            \'elapsed_seconds\': elapsed,\n            \'completion_tokens\': sum(row.get(\'completion_tokens\') or 0 for row in successes),\n            \'aggregate_completion_tokens_per_second\':\n                sum(row.get(\'completion_tokens\') or 0 for row in successes)/max(0.001, elapsed),\n            \'http_p50_seconds\': quantile([r[\'http_seconds\'] for r in successes], 0.5),\n            \'http_p95_seconds\': quantile([r[\'http_seconds\'] for r in successes], 0.95),\n            \'queue_p95_seconds\': quantile([r[\'queue_seconds\'] for r in rows], 0.95),\n            \'distinct_states_served\': len({r[\'state\'] for r in rows}),\n            \'unserved_states\': population-len({r[\'state\'] for r in rows}),\n            \'note\': \'Unserved states are pending-load scaffolding, not failed or solved games\',\n            \'rows\': sorted(rows, key=lambda r: r[\'case\'])}\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--base-url\', default=\'http://127.0.0.1:8001/v1\')\n    parser.add_argument(\'--model\', default=\'flashnext\')\n    parser.add_argument(\'--tokenizer\', required=True)\n    parser.add_argument(\'--output\', required=True)\n    args = parser.parse_args()\n    base = local_url(args.base_url)\n    root = base[:-3] if base.endswith(\'/v1\') else base\n    output = Path(args.output)\n    output.mkdir(parents=True, exist_ok=True)\n    from transformers import AutoTokenizer\n    import requests\n    config = ProbeConfig().validate()\n    tokenizer = AutoTokenizer.from_pretrained(args.tokenizer, local_files_only=True,\n                                              trust_remote_code=True)\n    plan = shape_plan(config)\n    # Preparing all inputs happens before phase timing; no model weights load.\n    payloads = [[build_payload(tokenizer, case, phase, args.model, config.populations[phase]) for case in plan]\n                for phase in range(len(config.populations))]\n    manifest = {\'scope\': __doc__, \'config\': asdict(config), \'request_plan\': plan,\n                \'official_score\': None, \'authorizes_submission\': False,\n                \'weights_loaded_by_probe\': False, \'server_launched_by_probe\': False}\n    (output/\'probe_manifest.json\').write_text(json.dumps(manifest, indent=2))\n    def snapshot(name):\n        try:\n            response = requests.get(root+\'/metrics\', timeout=(2, 3))\n            response.raise_for_status()\n            (output/(name+\'.prom\')).write_text(response.text)\n        except Exception as exc:\n            (output/(name+\'.error.txt\')).write_text(type(exc).__name__+\': \'+str(exc))\n    def post(payload, timeout):\n        response = requests.post(base+\'/chat/completions\', json=payload,\n                                 timeout=(min(2, timeout), timeout))\n        response.raise_for_status()\n        body = response.json()\n        usage = body.get(\'usage\') or {}\n        return {\'ok\': bool(body.get(\'choices\')), \'http_status\': response.status_code,\n                \'completion_tokens\': int(usage.get(\'completion_tokens\') or 0),\n                \'prompt_tokens\': int(usage.get(\'prompt_tokens\') or 0),\n                \'finish_reason\': (body.get(\'choices\') or [{}])[0].get(\'finish_reason\')}\n    warm_payload, _ = build_payload(tokenizer,\n        {\'case\': 0, \'target_input_tokens\': 1024, \'max_output_tokens\': 16, \'seed\': config.seed},\n        -1, args.model, 1)\n    warm_start = time.monotonic()\n    warm = post(warm_payload, 60.0)\n    manifest[\'warmup\'] = {**warm, \'elapsed_seconds\': time.monotonic()-warm_start}\n    (output/\'probe_manifest.json\').write_text(json.dumps(manifest, indent=2))\n    if not warm[\'ok\']:\n        raise RuntimeError(\'Warmup returned no choices; refusing timed phases\')\n    reports = []\n    started = time.monotonic()\n    for i, population in enumerate(config.populations):\n        snapshot(f\'phase{i}-before\')\n        report = run_phase(population, payloads[i], config, post)\n        report[\'phase\'] = i\n        reports.append(report)\n        (output/f\'phase{i}.json\').write_text(json.dumps(report, indent=2))\n        snapshot(f\'phase{i}-after\')\n        print(json.dumps({k: v for k, v in report.items() if k != \'rows\'}), flush=True)\n        if not report[\'valid_transport_comparison\']:\n            # Fail fast: more load would consume GPU without a comparable pair.\n            break\n    metrics_count = len(list(output.glob(\'phase*.prom\')))\n    result = {**manifest, \'phases\': reports, \'elapsed_seconds\': time.monotonic()-started,\n              \'raw_metrics_snapshot_count\': metrics_count,\n              \'cache_metrics_available\': metrics_count == 2*len(reports),\n              \'valid_transport_comparison\': len(reports) == 4 and all(\n                  r[\'valid_transport_comparison\'] for r in reports),\n              \'limitations\': [\'Synthetic text; no game policies, image decoding or hidden score\',\n                             \'FIFO ten-slot transport gate; D-prime scheduler is not reproduced\',\n                             \'25-state prefixes are reused;110-state phase admits64 unique states\',\n                             \'Repeated synthetic prefixes isolate working-set cache cost; real game histories differ\',\n                             \'Short2K/8K/16K probe cannot establish near-limit128K or long-run stability\',\n                             \'Compare only phases whose returned prompt/output distributions match\']}\n    (output/\'probe_summary.json\').write_text(json.dumps(result, indent=2))\n    return 0 if result[\'valid_transport_comparison\'] else 2\n\n\nif __name__ == \'__main__\':\n    raise SystemExit(main())\n')
assert hashlib.sha256(helper_path.read_bytes()).hexdigest() == '76c1d2aeba94c170eddc9ebbf303e5b7413dfd65240dc55f22745a4cbd9a00a0'
runtime_env = dict(os.environ)
runtime_env.update({'USE_TORCH':'0', 'USE_TF':'0', 'USE_FLAX':'0',
    'CUDA_VISIBLE_DEVICES':'', 'HF_HUB_OFFLINE':'1', 'TRANSFORMERS_OFFLINE':'1',
    'HF_DATASETS_OFFLINE':'1', 'PIP_NO_INDEX':'1', 'PIP_DISABLE_PIP_VERSION_CHECK':'1',
    'TOKENIZERS_PARALLELISM':'false', 'PYTHONNOUSERSITE':'1'})
runtime_env.pop('PYTHONPATH', None)
# Installed dependencies are disposable setup, not notebook output artifacts.
target = Path('/tmp/arc3-cpu-tokenizer-site-packages')
commands = [
    ('dependency-preparation', [sys.executable, '-u', str(installer), str(wheel_roots[0]), str(target), str(work/'dependencies.json')], 210),
    ('real-tokenizer-smoke', [sys.executable, '-u', str(runner), str(target), str(models[0]), str(helper_path), str(work/'tokenizer_smoke.json')], 120),
]
for label, command, timeout in commands:
    print(label, flush=True)
    result = subprocess.run(command, env=runtime_env, text=True, capture_output=True, timeout=timeout)
    (work/(label+'.log')).write_text(result.stdout+'\n'+result.stderr)
    print(result.stdout[-16000:], flush=True)
    if result.returncode:
        print(result.stderr[-6000:], flush=True)
        raise RuntimeError(label+' failed; inspect saved CPU evidence')
report = json.loads((work/'tokenizer_smoke.json').read_text())
assert report['passed']
print('CPU tokenizer shape smoke passed. No GPU, server, HTTP or model weights used.', flush=True)
